# Eksploracja danych GIOŚ

## Importy i dostęp do kodu projektu

In [1]:
import sys

sys.path.append("../src")

In [2]:
from smogcast.ingest.gios import (
    get_all_stations,
    get_archival_data_by_sensor,
)

In [3]:
from smogcast.ingest.gios import generate_date_ranges

## Pobranie stacji GIOŚ

In [4]:
stations = get_all_stations()

In [5]:
len(stations)

288

In [6]:
stations[0]

{'Identyfikator stacji': 11,
 'Kod stacji': 'DsCzerStraza',
 'Nazwa stacji': 'Czerniawa',
 'WGS84 φ N': '50.912475',
 'WGS84 λ E': '15.312190',
 'Identyfikator miasta': 142,
 'Nazwa miasta': 'Czerniawa',
 'Gmina': 'Świeradów-Zdrój',
 'Powiat': 'lubański',
 'Województwo': 'DOLNOŚLĄSKIE',
 'Ulica': 'ul. Strażacka 7'}

In [7]:
from smogcast.processing.filters import filter_silesian_stations

In [8]:
silesian_stations = filter_silesian_stations(stations)

In [9]:
len(silesian_stations)

31

In [10]:
[
    (
        station["Identyfikator stacji"],
        station["Nazwa stacji"],
        station["Nazwa miasta"],
    )
    for station in silesian_stations
]

[(789, 'Bielsko-Biała, ul. Kossak-Szczuckiej', 'Bielsko-Biała'),
 (791, 'Bielsko-Biała, ul. Sternicza', 'Bielsko-Biała'),
 (798, 'Częstochowa, Al. AK/Jana Pawła II', 'Częstochowa'),
 (800, 'Częstochowa, ul. Baczyńskiego', 'Częstochowa'),
 (804, 'Częstochowa, ul. Zana', 'Częstochowa'),
 (805, 'Dąbrowa Górnicza, ul. Tysiąclecia', 'Dąbrowa Górnicza'),
 (809, 'Gliwice, ul. Mewy', 'Gliwice'),
 (810, 'Godów, ul. Gliniki', 'Godów'),
 (814, 'Katowice, ul. Kossutha', 'Katowice'),
 (818, 'Knurów, ul. Jedności Narodowej', 'Knurów'),
 (825, 'Myszków, ul. Miedziana', 'Myszków'),
 (834, 'Rybnik, ul. Borki', 'Rybnik'),
 (837, 'Sosnowiec, ul. Lubelska', 'Sosnowiec'),
 (839, 'Tarnowskie Góry, ul. Litewska', 'Tarnowskie Góry'),
 (841, 'Tychy, ul. Tołstoja', 'Tychy'),
 (842, 'Ustroń, ul. Sanatoryjna', 'Ustroń'),
 (845, 'Wodzisław Śląski, ul. Gałczyńskiego', 'Wodzisław Śląski'),
 (853, 'Złoty Potok, Leśniczówka', 'Złoty Potok'),
 (856, 'Żywiec, ul. Kopernika', 'Żywiec'),
 (9000, 'Czerwionka-Leszczyny, ul.

## Test na jednej stacji

## Zapis surowych danych

In [2]:
import sys

sys.path.append("../src")

In [3]:
from smogcast.ingest.gios import (
    get_all_stations,
    get_archival_data_by_sensor,
    get_data_for_sensors,
)

In [4]:
from smogcast.ingest.gios import generate_date_ranges

In [5]:
from smogcast.processing.filters import filter_silesian_stations

In [6]:
import pandas as pd

In [7]:
stations = get_all_stations()

In [8]:
silesian_stations = filter_silesian_stations(stations)

In [8]:
station_id = 814

In [9]:
import httpx

In [10]:
url = f"https://api.gios.gov.pl/pjp-api/v1/rest/station/sensors/{station_id}"

response = httpx.get(url, timeout=30.0)
response.raise_for_status()

sensor_data = response.json()

In [11]:
sensors = sensor_data["Lista stanowisk pomiarowych dla podanej stacji"]

In [12]:
pm_sensors = [
    sensor
    for sensor in sensors
    if sensor.get("Wskaźnik") in {
        "pył zawieszony PM10",
        "pył zawieszony PM2.5",
    }
]

In [13]:
sensor_ids = [
    sensor["Identyfikator stanowiska"]
    for sensor in pm_sensors
]

In [14]:
sensor_ids

[5376, 5377, 5378, 5379]

In [15]:
all_measurements, failed_ranges = get_data_for_sensors(
    sensor_ids,
    "2026-08-20 10:00",
    "2026-08-20 12:00",
    days_per_range=1,
)

Rozpoczynam pobieranie sensora 5376
Pobieram sensor 5376: 2026-08-20 10:00 - 2026-08-20 12:00
Rozpoczynam pobieranie sensora 5377
Pobieram sensor 5377: 2026-08-20 10:00 - 2026-08-20 12:00
Rozpoczynam pobieranie sensora 5378
Pobieram sensor 5378: 2026-08-20 10:00 - 2026-08-20 12:00
Rozpoczynam pobieranie sensora 5379
Pobieram sensor 5379: 2026-08-20 10:00 - 2026-08-20 12:00


In [23]:
df = pd.DataFrame(all_measurements)

In [24]:
df

,Nazwa stacji,Kod stanowiska,Data,Wartość
0,"Katowice, ul. Kossutha",SlKatoKossut-PM10-1g,2026-08-20 10:00:00,10.9
1,"Katowice, ul. Kossutha",SlKatoKossut-PM10-1g,2026-08-20 11:00:00,10.6
2,"Katowice, ul. Kossutha",SlKatoKossut-PM10-1g,2026-08-20 12:00:00,11.4
3,"Katowice, ul. Kossutha",SlKatoKossut-PM2.5-1g,2026-08-20 10:00:00,6.1
4,"Katowice, ul. Kossutha",SlKatoKossut-PM2.5-1g,2026-08-20 11:00:00,5.5
5,"Katowice, ul. Kossutha",SlKatoKossut-PM2.5-1g,2026-08-20 12:00:00,5.2


In [25]:
df.to_parquet(
    "../data/raw/gios_pm_measurements.parquet",
    index=False,
)

In [26]:
pd.read_parquet("../data/raw/gios_pm_measurements.parquet").head()

,Nazwa stacji,Kod stanowiska,Data,Wartość
0,"Katowice, ul. Kossutha",SlKatoKossut-PM10-1g,2026-08-20 10:00:00,10.9
1,"Katowice, ul. Kossutha",SlKatoKossut-PM10-1g,2026-08-20 11:00:00,10.6
2,"Katowice, ul. Kossutha",SlKatoKossut-PM10-1g,2026-08-20 12:00:00,11.4
3,"Katowice, ul. Kossutha",SlKatoKossut-PM2.5-1g,2026-08-20 10:00:00,6.1
4,"Katowice, ul. Kossutha",SlKatoKossut-PM2.5-1g,2026-08-20 11:00:00,5.5


## Lista slaskich stacji z pm10 albo pm2.5

In [10]:
stations = get_all_stations()

In [11]:
silesian_stations = filter_silesian_stations(stations)

In [12]:
import httpx
import time

In [13]:
stations_with_pm = []

In [14]:
for station in silesian_stations:
    station_id = station["Identyfikator stacji"]

    url = (
        "https://api.gios.gov.pl/pjp-api/"
        f"v1/rest/station/sensors/{station_id}"
    )

    response = httpx.get(url, timeout=30.0)
    response.raise_for_status()

    sensor_data = response.json()

    sensors = sensor_data[
        "Lista stanowisk pomiarowych dla podanej stacji"
    ]

    pm_sensors = [
        sensor
        for sensor in sensors
        if sensor.get("Wskaźnik") in {
            "pył zawieszony PM10",
            "pył zawieszony PM2.5",
        }
    ]

    if pm_sensors:
        stations_with_pm.append(
            {
                "station_id": station_id,
                "station_name": station["Nazwa stacji"],
                "city": station["Nazwa miasta"],
                "pm_sensors": pm_sensors,
            }
        )

    time.sleep(1)

In [15]:
len(stations_with_pm)

30

In [16]:
import pandas as pd

In [17]:
stations_pm_df = pd.DataFrame(
    [
        {
            "station_id": station["station_id"],
            "station_name": station["station_name"],
            "city": station["city"],
            "pm10_count": sum(
                sensor.get("Wskaźnik") == "pył zawieszony PM10"
                for sensor in station["pm_sensors"]
            ),
            "pm25_count": sum(
                sensor.get("Wskaźnik") == "pył zawieszony PM2.5"
                for sensor in station["pm_sensors"]
            ),
        }
        for station in stations_with_pm
    ]
)

stations_pm_df

,station_id,station_name,city,pm10_count,pm25_count
0,789,"Bielsko-Biała, ul. Kossak-Szczuckiej",Bielsko-Biała,2,0
1,791,"Bielsko-Biała, ul. Sternicza",Bielsko-Biała,0,1
2,798,"Częstochowa, Al. AK/Jana Pawła II",Częstochowa,1,0
3,800,"Częstochowa, ul. Baczyńskiego",Częstochowa,2,0
4,804,"Częstochowa, ul. Zana",Częstochowa,0,1
5,805,"Dąbrowa Górnicza, ul. Tysiąclecia",Dąbrowa Górnicza,2,0
6,809,"Gliwice, ul. Mewy",Gliwice,1,1
7,810,"Godów, ul. Gliniki",Godów,1,1
8,814,"Katowice, ul. Kossutha",Katowice,2,2
9,818,"Knurów, ul. Jedności Narodowej",Knurów,1,0


In [18]:
len(stations_pm_df)

30

In [19]:
stations_pm_df.to_parquet(
    "../data/raw/gios_silesian_pm_stations.parquet",
    index=False,
)

In [20]:
stations_pm_df.to_csv(
    "../data/raw/gios_silesian_pm_stations.csv",
    index=False,
)

## Lista polskich stacji z pm10 albo pm2.5

In [44]:
stations_with_pm_poland = []

In [45]:
for station in stations:
    station_id = station["Identyfikator stacji"]

    url = (
        "https://api.gios.gov.pl/pjp-api/"
        f"v1/rest/station/sensors/{station_id}"
    )

    response = httpx.get(url, timeout=30.0)
    response.raise_for_status()

    sensor_data = response.json()

    sensors = sensor_data[
        "Lista stanowisk pomiarowych dla podanej stacji"
    ]

    pm_sensors = [
        sensor
        for sensor in sensors
        if sensor.get("Wskaźnik") in {
            "pył zawieszony PM10",
            "pył zawieszony PM2.5",
        }
    ]

    if pm_sensors:
        stations_with_pm_poland.append(
            {
                "station_id": station_id,
                "station_name": station["Nazwa stacji"],
                "city": station["Nazwa miasta"],
                "voivodeship": station["Województwo"],
                "pm_sensors": pm_sensors,
            }
        )

    time.sleep(1)

In [46]:
len(stations_with_pm_poland)

265

In [47]:
stations_pm_poland_df = pd.DataFrame(
    [
        {
            "station_id": station["station_id"],
            "station_name": station["station_name"],
            "city": station["city"],
            "voivodeship": station["voivodeship"],
            "pm10_count": sum(
                sensor.get("Wskaźnik") == "pył zawieszony PM10"
                for sensor in station["pm_sensors"]
            ),
            "pm25_count": sum(
                sensor.get("Wskaźnik") == "pył zawieszony PM2.5"
                for sensor in station["pm_sensors"]
            ),
        }
        for station in stations_with_pm_poland
    ]
)

In [48]:
stations_pm_poland_df

,station_id,station_name,city,voivodeship,pm10_count,pm25_count
0,11,Czerniawa,Czerniawa,DOLNOŚLĄSKIE,1,0
1,16,"Dzierżoniów, ul. Piłsudskiego",Dzierżoniów,DOLNOŚLĄSKIE,1,0
2,38,"Kłodzko, ul. Szkolna",Kłodzko,DOLNOŚLĄSKIE,1,1
3,52,"Legnica, Al. Rzeczypospolitej",Legnica,DOLNOŚLĄSKIE,2,1
4,70,"Oława, ul. Żołnierzy Armii Krajowej",Oława,DOLNOŚLĄSKIE,2,1
...,...,...,...,...,...,...
260,20568,"Solec-Zdrój, ul. Leśna",Solec Zdrój,ŚWIĘTOKRZYSKIE,2,1
261,20590,ul. Sportowa 15,Kobierzyce,DOLNOŚLĄSKIE,1,0
262,20591,"Solec Kujawski, ul. Tartaczna 10",Solec Kujawski,KUJAWSKO-POMORSKIE,1,1
263,20607,"Tczew, ul. Czyżykowska",Tczew,POMORSKIE,1,1


In [49]:
stations_pm_poland_df["voivodeship"].value_counts()

voivodeship
ŚLĄSKIE                30
MAŁOPOLSKIE            28
DOLNOŚLĄSKIE           24
MAZOWIECKIE            24
ŁÓDZKIE                19
KUJAWSKO-POMORSKIE     18
PODKARPACKIE           18
WIELKOPOLSKIE          18
POMORSKIE              14
ŚWIĘTOKRZYSKIE         14
LUBELSKIE              12
LUBUSKIE               10
OPOLSKIE               10
ZACHODNIOPOMORSKIE     10
PODLASKIE               9
WARMIŃSKO-MAZURSKIE     7
Name: count, dtype: int64

In [52]:
stations_pm_poland_df.to_parquet(
    "../data/raw/gios_poland_pm_stations.parquet",
    index=False,
)

In [53]:
stations_pm_poland_df.to_csv(
    "../data/raw/gios_poland_pm_stations.csv",
    index=False,
)

### sprawdzenie zakresów dat i ewentualnych luk

In [56]:
df["Data"] = pd.to_datetime(df["Data"])

In [57]:
min_date = df["Data"].min()
max_date = df["Data"].max()

print("Najstarszy pomiar:", min_date)
print("Najnowszy pomiar:", max_date)

Najstarszy pomiar: 2026-08-20 10:00:00
Najnowszy pomiar: 2026-08-20 12:00:00


## Lista slaskich stacji- 3 lata

In [21]:
sensor_records = []

for station in stations_with_pm:
    for sensor in station["pm_sensors"]:
        indicator = sensor.get("Wskaźnik")

        if indicator == "pył zawieszony PM10":
            parameter = "PM10"
        elif indicator == "pył zawieszony PM2.5":
            parameter = "PM2.5"
        else:
            continue

        sensor_records.append(
            {
                "station_id": station["station_id"],
                "sensor_id": sensor["Identyfikator stanowiska"],
                "parameter": parameter,
            }
        )

In [22]:
sensor_records[:10]

[{'station_id': 789, 'sensor_id': 5167, 'parameter': 'PM10'},
 {'station_id': 789, 'sensor_id': 5168, 'parameter': 'PM10'},
 {'station_id': 791, 'sensor_id': 5177, 'parameter': 'PM2.5'},
 {'station_id': 798, 'sensor_id': 5232, 'parameter': 'PM10'},
 {'station_id': 800, 'sensor_id': 5256, 'parameter': 'PM10'},
 {'station_id': 800, 'sensor_id': 5257, 'parameter': 'PM10'},
 {'station_id': 804, 'sensor_id': 5271, 'parameter': 'PM2.5'},
 {'station_id': 805, 'sensor_id': 5286, 'parameter': 'PM10'},
 {'station_id': 805, 'sensor_id': 5287, 'parameter': 'PM10'},
 {'station_id': 809, 'sensor_id': 5312, 'parameter': 'PM10'}]

In [23]:
len(sensor_records)

52

In [24]:
date_from = "2023-08-20 00:00"
date_to = "2026-08-20 23:00"

In [25]:
from smogcast.ingest.gios import get_data_for_sensor_records

In [26]:
all_measurements, failed_ranges = get_data_for_sensor_records(
    sensor_records,
    date_from,
    date_to,
    days_per_range=30,
)

Rozpoczynam: stacja 789, sensor 5167, PM10
Pobieram sensor 5167: 2023-08-20 00:00 - 2023-09-19 00:00
Timeout dla sensora 5167: 2023-08-20 00:00 - 2023-09-19 00:00. Próba 1/3
Pobieram sensor 5167: 2023-09-19 00:00 - 2023-10-19 00:00
Pobieram sensor 5167: 2023-10-19 00:00 - 2023-11-18 00:00
Pobieram sensor 5167: 2023-11-18 00:00 - 2023-12-18 00:00
Pobieram sensor 5167: 2023-12-18 00:00 - 2024-01-17 00:00
Pobieram sensor 5167: 2024-01-17 00:00 - 2024-02-16 00:00
Pobieram sensor 5167: 2024-02-16 00:00 - 2024-03-17 00:00
Pobieram sensor 5167: 2024-03-17 00:00 - 2024-04-16 00:00
Pobieram sensor 5167: 2024-04-16 00:00 - 2024-05-16 00:00
Pobieram sensor 5167: 2024-05-16 00:00 - 2024-06-15 00:00
Pobieram sensor 5167: 2024-06-15 00:00 - 2024-07-15 00:00
Pobieram sensor 5167: 2024-07-15 00:00 - 2024-08-14 00:00
Pobieram sensor 5167: 2024-08-14 00:00 - 2024-09-13 00:00
Pobieram sensor 5167: 2024-09-13 00:00 - 2024-10-13 00:00
Pobieram sensor 5167: 2024-10-13 00:00 - 2024-11-12 00:00
Pobieram senso

KeyboardInterrupt: 

In [4]:
import sys
from pathlib import Path

import pandas as pd

sys.path.append(str(Path.cwd().parent / "src"))

from smogcast.ingest.gios import get_data_for_sensor_records

In [5]:
CURRENT_GIOS_PATH = "../data/raw/gios_pm_measurements_1y.parquet"
OLDER_GIOS_PATH = "../data/raw/gios_pm_measurements_older.parquet"
FULL_GIOS_PATH = "../data/raw/gios_pm_measurements_3y.parquet"

DATE_FROM = "2023-08-20 00:00"
DATE_TO = "2025-08-19 23:00"

In [6]:
current_gios = pd.read_parquet(CURRENT_GIOS_PATH)

current_gios.head()

,station_id,sensor_id,station_name,city,voivodeship,Parametr,Data,Wartość,Kod stanowiska
0,117,669,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,PM10,2025-08-20,29.9,DsWrocWybCon-PM10-24g
1,117,669,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,PM10,2025-08-21,23.0,DsWrocWybCon-PM10-24g
2,117,669,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,PM10,2025-08-22,21.2,DsWrocWybCon-PM10-24g
3,117,669,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,PM10,2025-08-23,23.5,DsWrocWybCon-PM10-24g
4,117,669,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,PM10,2025-08-24,10.2,DsWrocWybCon-PM10-24g


In [7]:
current_gios.shape

(225014, 9)

In [8]:
current_gios.columns.tolist()

['station_id',
 'sensor_id',
 'station_name',
 'city',
 'voivodeship',
 'Parametr',
 'Data',
 'Wartość',
 'Kod stanowiska']

In [9]:
sensor_records_df = (
    current_gios[
        [
            "station_id",
            "sensor_id",
            "station_name",
            "city",
            "voivodeship",
            "Parametr",
        ]
    ]
    .drop_duplicates()
    .rename(
        columns={
            "Parametr": "parameter",
        }
    )
    .reset_index(drop=True)
)

sensor_records = sensor_records_df.to_dict(
    orient="records"
)

print("Liczba sensorów:", len(sensor_records))

sensor_records_df.head()

Liczba sensorów: 43


,station_id,sensor_id,station_name,city,voivodeship,parameter
0,117,669,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,PM10
1,117,670,"Wrocław, wyb. Conrada-Korzeniowskiego",Wrocław,DOLNOŚLĄSKIE,PM2.5
2,156,965,"Bydgoszcz, Pl. Poznański",Bydgoszcz,KUJAWSKO-POMORSKIE,PM10
3,156,966,"Bydgoszcz, Pl. Poznański",Bydgoszcz,KUJAWSKO-POMORSKIE,PM2.5
4,258,1832,"Lublin, ul. Śliwińskiego",Lublin,LUBELSKIE,PM10


In [10]:
sensor_records_df["station_id"].nunique()

16

In [12]:
import contextlib
import io

from tqdm.auto import tqdm

/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [13]:
older_measurements = []
failed_ranges = []

for sensor in tqdm(
    sensor_records,
    desc="Pobieranie danych GIOŚ",
    unit="sensor",
):
    # Wycisza komunikaty print() z funkcji ingestu.
    with contextlib.redirect_stdout(io.StringIO()):
        measurements, sensor_failed_ranges = get_data_for_sensor_records(
            sensor_records=[sensor],
            date_from=DATE_FROM,
            date_to=DATE_TO,
            days_per_range=30,
        )

    older_measurements.extend(measurements)
    failed_ranges.extend(sensor_failed_ranges)

Pobieranie danych GIOŚ: 100%|███████████████| 43/43 [36:54<00:00, 51.50s/sensor]


In [14]:
older_gios = pd.DataFrame(older_measurements)

older_gios.shape

(434784, 10)

In [15]:
older_gios["Data"] = pd.to_datetime(older_gios["Data"])

print(
    older_gios["Data"].min(),
    "→",
    older_gios["Data"].max(),
)

2023-08-20 00:00:00 → 2025-08-19 23:00:00


In [16]:
print("Nieudane zakresy:", len(failed_ranges))

Nieudane zakresy: 0


In [17]:
older_gios.to_parquet(
    "../data/raw/gios_pm_measurements_older.parquet",
    index=False,
)

In [18]:
current_gios = pd.read_parquet(
    "../data/raw/gios_pm_measurements_1y.parquet"
)

In [19]:
current_gios["Data"] = pd.to_datetime(current_gios["Data"])

In [20]:
gios_3y = pd.concat(
    [
        older_gios,
        current_gios,
    ],
    ignore_index=True,
)

In [21]:
before = len(gios_3y)

gios_3y = (
    gios_3y
    .drop_duplicates(
        subset=["sensor_id", "Data"]
    )
    .reset_index(drop=True)
)

after = len(gios_3y)

print("Przed:", before)
print("Po:", after)
print("Duplikaty usunięte:", before - after)

Przed: 659798
Po: 659754
Duplikaty usunięte: 44


In [22]:
print(
    "Zakres:",
    gios_3y["Data"].min(),
    "→",
    gios_3y["Data"].max(),
)

print("Rekordów:", len(gios_3y))
print("Stacji:", gios_3y["station_id"].nunique())
print("Sensorów:", gios_3y["sensor_id"].nunique())

Zakres: 2023-08-20 00:00:00 → 2026-08-20 23:00:00
Rekordów: 659754
Stacji: 16
Sensorów: 43


In [23]:
gios_3y.to_parquet(
    "../data/raw/gios_pm_measurements_3y.parquet",
    index=False,
)